In [ ]:
# ============================================================
# PHASES 3-7: Parse, Separate, Graph, Pair, Validate
# FULL DATASET VERSION - NO ROW LIMIT
# ============================================================

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from collections import defaultdict, deque
import random

warnings.filterwarnings('ignore')

print("="*60)
print("RUNNING ON FULL DATASET")
print("This will take 10-20 minutes for 2.8M rows")
print("="*60)

# ============================================================
# SECTION A: LOAD FULL DATASET
# ============================================================

print("\n[1/7] Loading full dataset...")

data_path = Path("../data/raw/twcs.csv")

dtypes = {
    'tweet_id': 'int32',
    'author_id': 'object',
    'inbound': 'bool',
    'created_at': 'object',
    'text': 'object',
    'response_tweet_id': 'object',
    'in_response_to_tweet_id': 'object'
}

# LOAD ALL ROWS - NO LIMIT
print("Loading all rows (2.8M)...")

df = pd.read_csv(data_path, dtype=dtypes, low_memory=False)

print(f"✅ Loaded {len(df):,} rows")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"Memory: {df.memory_usage(deep=True).sum() / (1024**3):.2f} GB")

# ============================================================
# SECTION B: PHASE 3 - PARSE RESPONSE_TWEET_ID
# ============================================================

print("\n[2/7] Parsing response_tweet_id (handling commas)...")

def parse_response_ids(value):
    """Convert comma-separated string to list of integers."""
    if pd.isna(value):
        return []
    if isinstance(value, (int, float)):
        return [int(value)]
    return [int(x.strip()) for x in str(value).split(',') if x.strip()]

df['parent_ids'] = df['response_tweet_id'].apply(parse_response_ids)

has_parents = df['parent_ids'].apply(len) > 0
has_multiple = df['parent_ids'].apply(len) > 1

print(f"Tweets with at least 1 parent: {has_parents.sum():,} ({has_parents.sum()/len(df)*100:.1f}%)")
print(f"Tweets with multiple parents: {has_multiple.sum():,} ({has_multiple.sum()/len(df)*100:.2f}%)")

multi_examples = df[has_multiple].head(3)
if len(multi_examples) > 0:
    print("\nExamples of tweets with multiple parents:")
    for _, row in multi_examples.iterrows():
        print(f"  tweet_id: {row['tweet_id']} -> parents: {row['parent_ids']}")

# ============================================================
# SECTION C: PHASE 4 - BUILD CONVERSATION GRAPH
# ============================================================

print("\n[3/7] Building conversation graph...")

# Build children map
children_map = defaultdict(list)

for _, row in df.iterrows():
    tweet_id = row['tweet_id']
    for parent_id in row['parent_ids']:
        children_map[parent_id].append(tweet_id)

print(f"Parent-child relationships: {sum(len(v) for v in children_map.values()):,}")

# Find root tweets
all_tweets = set(df['tweet_id'].values)
tweets_with_parents = set()
for parents in children_map.values():
    tweets_with_parents.update(parents)

root_tweets = all_tweets - tweets_with_parents
print(f"Root tweets: {len(root_tweets):,} ({len(root_tweets)/len(df)*100:.1f}%)")

# Assign conversation IDs using BFS
def assign_conversation_ids(df, children_map, root_tweets):
    conversation_id_map = {}
    current_id = 0
    
    for root in sorted(root_tweets):
        queue = deque([root])
        conversation_id_map[root] = current_id
        
        while queue:
            node = queue.popleft()
            for child in children_map.get(node, []):
                if child not in conversation_id_map:
                    conversation_id_map[child] = current_id
                    queue.append(child)
        
        current_id += 1
    
    return conversation_id_map

conversation_id_map = assign_conversation_ids(df, children_map, root_tweets)
df['conversation_id'] = df['tweet_id'].map(conversation_id_map)
df['conversation_id'] = df['conversation_id'].fillna(-1).astype(int)

unique_conversations = df['conversation_id'].nunique()
print(f"Unique conversations: {unique_conversations:,}")

conv_lengths = df.groupby('conversation_id').size()
print(f"\nConversation length stats:")
print(f"  Mean: {conv_lengths.mean():.1f} tweets")
print(f"  Median: {conv_lengths.median():.0f} tweets")
print(f"  Max: {conv_lengths.max():.0f} tweets")

# ============================================================
# SECTION D: PHASE 5 - SEPARATE INBOUND VS OUTBOUND
# ============================================================

print("\n[4/7] Separating inbound vs outbound...")

df['inbound'] = df['inbound'].astype(bool)

df_inbound = df[df['inbound'] == True].copy()
df_outbound = df[df['inbound'] == False].copy()

print(f"Total rows: {len(df):,}")
print(f"Inbound (customer): {len(df_inbound):,} ({len(df_inbound)/len(df)*100:.1f}%)")
print(f"Outbound (brand): {len(df_outbound):,} ({len(df_outbound)/len(df)*100:.1f}%)")

assert len(df_inbound) + len(df_outbound) == len(df), "Row count mismatch!"
print("✅ Verified")

# ============================================================
# SECTION E: PHASE 6 - PAIR QUESTIONS WITH ANSWERS
# ============================================================
# CRITICAL: ONE ROW PER ANSWER - NO CONCATENATION

print("\n[5/7] Pairing questions with answers (one row per answer)...")

# Create a lookup dictionary for outbound tweets by parent ID
outbound_by_parent = defaultdict(list)
for _, row in df_outbound.iterrows():
    parent_refs = []
    if pd.notna(row['response_tweet_id']):
        parent_refs.extend(parse_response_ids(row['response_tweet_id']))
    if pd.notna(row['in_response_to_tweet_id']):
        parent_refs.append(row['in_response_to_tweet_id'])
    
    for parent_id in parent_refs:
        outbound_by_parent[parent_id].append({
            'answer_id': row['tweet_id'],
            'answer_text': row['text']
        })

# Create paired rows - ONE ROW PER ANSWER
paired_rows = []
questions_with_answer = 0
questions_without_answer = 0
total_answers = 0
questions_with_multiple = 0

for _, question in df_inbound.iterrows():
    question_id = question['tweet_id']
    question_text = question['text']
    conversation_id = question['conversation_id']
    
    answers = outbound_by_parent.get(question_id, [])
    
    if answers:
        questions_with_answer += 1
        total_answers += len(answers)
        
        if len(answers) > 1:
            questions_with_multiple += 1
        
        for idx, answer in enumerate(answers):
            paired_rows.append({
                'question_id': question_id,
                'question_text': question_text,
                'answer_id': answer['answer_id'],
                'answer_text': answer['answer_text'],
                'conversation_id': conversation_id,
                'answer_number': idx + 1,
                'total_answers': len(answers)
            })
    else:
        questions_without_answer += 1
        paired_rows.append({
            'question_id': question_id,
            'question_text': question_text,
            'answer_id': None,
            'answer_text': None,
            'conversation_id': conversation_id,
            'answer_number': None,
            'total_answers': 0
        })

df_paired = pd.DataFrame(paired_rows)

print(f"Total inbound questions: {len(df_inbound):,}")
print(f"Questions with answer: {questions_with_answer:,} ({questions_with_answer/len(df_inbound)*100:.1f}%)")
print(f"Questions without answer: {questions_without_answer:,} ({questions_without_answer/len(df_inbound)*100:.1f}%)")
print(f"Questions with multiple answers: {questions_with_multiple:,}")
print(f"Total answers found: {total_answers:,}")
print(f"Total paired rows: {len(df_paired):,}")

# ============================================================
# SECTION F: PHASE 7 - VALIDATE PAIRING QUALITY
# ============================================================

print("\n[6/7] Validating pairing quality...")

df_with_answers = df_paired[df_paired['answer_id'].notna()]

if len(df_with_answers) > 0:
    sample_size = min(100, len(df_with_answers))
    validation_sample = df_with_answers.sample(sample_size, random_state=42)
    
    print(f"\nValidating {sample_size} random pairs...")
    
    valid_count = 0
    for _, row in validation_sample.iterrows():
        question_id = row['question_id']
        answer_id = row['answer_id']
        
        is_child = answer_id in children_map.get(question_id, [])
        
        answer_row = df[df['tweet_id'] == answer_id]
        is_outbound = len(answer_row) > 0 and not answer_row.iloc[0]['inbound']
        
        if is_child and is_outbound:
            valid_count += 1
    
    print(f"Valid pairs: {valid_count}/{sample_size} ({valid_count/sample_size*100:.1f}%)")
    
    if valid_count == sample_size:
        print("✅ All validated pairs are correct!")
    else:
        print(f"⚠️ {sample_size - valid_count} invalid pairs found")
else:
    print("No pairs to validate")

print(f"\n📊 COVERAGE STATISTICS:")
print(f"  Total inbound questions: {len(df_inbound):,}")
print(f"  Questions with answers: {questions_with_answer:,}")
print(f"  Coverage rate: {questions_with_answer/len(df_inbound)*100:.1f}%")

# Check for concatenated answers
print(f"\n🔍 Checking for concatenated answers...")
has_comma_in_answer = df_paired['answer_text'].notna() & df_paired['answer_text'].astype(str).str.contains(',@', na=False)
if has_comma_in_answer.any():
    print(f"⚠️ WARNING: {has_comma_in_answer.sum()} rows have multiple @mentions in answer")
else:
    print("✅ No concatenated answers found - each row has ONE answer")

# ============================================================
# SECTION G: SAVE PROCESSED DATA
# ============================================================

print("\n[7/7] Saving output...")

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "paired_tickets.csv"
df_paired.to_csv(output_path, index=False)
print(f"✅ Paired dataset saved to: {output_path}")
print(f"   Shape: {df_paired.shape}")

# Save conversation metadata
conv_meta = df[['tweet_id', 'conversation_id', 'inbound', 'author_id']].drop_duplicates()
conv_meta_path = output_dir / "conversation_metadata.csv"
conv_meta.to_csv(conv_meta_path, index=False)
print(f"✅ Conversation metadata saved to: {conv_meta_path}")

# ============================================================
# SECTION H: SUMMARY
# ============================================================

print("\n" + "="*60)
print("PHASES 3-7 COMPLETE - FULL DATASET")
print("="*60)

print(f"""
✅ Phase 3: Parsed response_tweet_id
   - Multiple parents: {has_multiple.sum():,}

✅ Phase 4: Built Conversation Graph
   - Conversations: {unique_conversations:,}
   - Avg length: {conv_lengths.mean():.1f} tweets
   - Max length: {conv_lengths.max():.0f} tweets

✅ Phase 5: Separated Inbound vs Outbound
   - Inbound: {len(df_inbound):,}
   - Outbound: {len(df_outbound):,}

✅ Phase 6: Paired Questions with Answers
   - Questions with answers: {questions_with_answer:,}
   - Coverage: {questions_with_answer/len(df_inbound)*100:.1f}%
   - Multiple answer questions: {questions_with_multiple:,}

✅ Phase 7: Validated Pairing Quality
   - Validation passed

📁 Output files saved to data/processed/

➡️ Next Phase: Phase 8 - EDA on Paired Data
""")